In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, date_format
import re
import shutil
from datetime import date

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Cost Price Report") \
    .getOrCreate()

# Load the Excel file into DataFrame (using the appropriate format)
filePath_primary = r'C:\Users\SYSTCS491\Documents\Monday CPCR\CPCR Primary Extract.xlsx'
filePath_secondary = r'C:\Users\SYSTCS491\Documents\Monday CPCR\CPCR Secondary Extract.xls'

primary_df = spark.read.format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .load(filePath_primary)

secondary_df = spark.read.format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .load(filePath_secondary)

# Filter data based on AttributeID
agreeItemNoList = primary_df.filter(col('AttributeID') == 'AgreementItemNumber')
agreeItemNetCostList = primary_df.filter(col('AttributeID') == 'AgreementItemNetCostPrice')
agreeItemProposedCostList = primary_df.filter(col('AttributeID') == 'ItemProposedCostPrice')
agreeCaseCountList = primary_df.filter(col('AttributeID') == 'AgreementItemCaseCount')
costStatusList = primary_df.filter(col('AttributeID') == 'CostStatus')

# Perform joins and create MasterList
master_df = agreeItemNoList.alias('a') \
    .join(agreeItemNetCostList.alias('b'), (col('a.ID') == col('b.ID')) & (col('a.ProductID') == col('b.ProductID')), 'left') \
    .join(agreeItemProposedCostList.alias('c'), (col('a.ID') == col('c.ID')) & (col('a.ProductID') == col('c.ProductID')), 'left') \
    .join(agreeCaseCountList.alias('d'), (col('a.ID') == col('d.ID')) & (col('a.ProductID') == col('d.ProductID')), 'left') \
    .join(costStatusList.alias('e'), (col('a.ID') == col('e.ID')) & (col('a.ProductID') == col('e.ProductID')), 'left') \
    .select(
        col('a.ID').alias('ID'),
        col('a.ProductID').alias('Item Supplier Site'),
        col('a.Value').alias('Product ID'),
        col('b.Value').alias('Current Cost Price'),
        col('c.Value').alias('Proposed Cost Price'),
        col('d.Value').alias('Case Count'),
        col('e.Value').alias('Cost Status')
    )

# Remove failed agreements
pattern = "Failed"
master_df = master_df.filter(~col('Cost Status').rlike(pattern))

# Load secondary data into DataFrame and perform final join
final_df = master_df.alias('m') \
    .join(secondary_df.alias('s'), col('m.ID') == col('s.<ID>'), 'left') \
    .select(
        col('m.ID'),
        col('m.Item Supplier Site'),
        col('m.Product ID'),
        col('m.Current Cost Price'),
        col('m.Proposed Cost Price'),
        col('m.Case Count'),
        col('m.Cost Status'),
        col('s.Trading Team'),
        col('s.Trader Team'),
        col('s.Status'),
        col('s.Cost Price Start Date'),
        col('s.RMS Supplier ID').alias('Supplier Number'),
        col('s.Supplier'),
        col('s.STEP Supplier Trading Unit ID'),
        col('s.Supplier Trading Unit Name'),
        col('s.RMS Supplier Trading Unit ID'),
        date_format(col('s.Complete Cost Price Change Request - On Entry TS').substr(1, 10), 'yyyy-MM-dd').alias('CPCR Entry Date')
    )

# Remove undesired statuses
final_df = final_df.filter(~(col('Status').isin(['Closed (Not Accepted)', 'Rejected'])))

# Separate lists for 'In Progress' and 'Completed'
in_progress_df = final_df.filter(col('Status') == 'In Progress')
completed_df = final_df.filter(col('Status') != 'In Progress')

# Convert to Excel and save
today = str(date.today())
output_filename = f'Cost Price Report - {today}.xlsx'
output_path = f"C:/Users/SYSTCS491/Documents/Monday CPCR/{output_filename}"

# Write to Excel using Pandas after converting back to Pandas DataFrame
in_progress_pd = in_progress_df.toPandas()
completed_pd = completed_df.toPandas()

with pd.ExcelWriter(output_path) as writer:
    completed_pd.to_excel(writer, sheet_name='Completed', index=False)
    in_progress_pd.to_excel(writer, sheet_name='In Progress', index=False)

print("Execution Completed!")
spark.stop()
